In [ ]:
from sentence_transformers import SentenceTransformer, util

# Load the pretrained model
model = SentenceTransformer('all-mpnet-base-v2')

# Example hand-crafted data (issue body + comments)
issue_body = "Hi, I've been using nmap for a long time to do scans for PTR records. Now I'd like to do the same using masscan for its speed. Is that possible?"

comments = [
    "Can you clarify what you mean by 'PTR records'? Are you referring to reverse DNS lookups?",
    "Did you try running the scan on subnet 192.168.1.0/24?",
    "What tool do you normally use for vulnerability scanning?"
]

# Encode the issue and comments
issue_embedding = model.encode(issue_body, convert_to_tensor=True)
comment_embeddings = model.encode(comments, convert_to_tensor=True)

# Compute cosine similarities
cosine_scores = util.cos_sim(issue_embedding, comment_embeddings)[0]

# Threshold for considering a question "clarifying"
threshold = 0.7

for idx, (comment, score) in enumerate(zip(comments, cosine_scores)):
    print(f"Comment {idx+1}: \"{comment}\"")
    print(f"Similarity score: {score:.4f}")
    if score >= threshold:
        print("-> Likely a clarifying question indicating ambiguity.\n")
    else:
        print("-> Likely unrelated or tangential question.\n")

/Users/harshdarji/Documents/Research-Internship-II/revised-ra2-iclr/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Comment 1: "Can you clarify what you mean by 'PTR records'? Are you referring to reverse DNS lookups?"
Similarity score: 0.4290
-> Likely unrelated or tangential question.

Comment 2: "Did you try running the scan on subnet 192.168.1.0/24?"
Similarity score: 0.2666
-> Likely unrelated or tangential question.

Comment 3: "What tool do you normally use for vulnerability scanning?"
Similarity score: 0.1485
-> Likely unrelated or tangential question.



In [4]:
import json
import os
import re
import nltk
from nltk.tokenize import wordpunct_tokenize as word_tokenize
from collections import defaultdict
import chardet

# Download only stopwords (we don't need 'punkt' since we're using wordpunct_tokenize)
nltk.download('stopwords')

# Define categories and their associated keywords/phrases
categories = {
    "Modal Verbs": ["can", "could", "would", "should", "may", "might", "will", "shall", "must"],
    "Interrogatives": ["who", "what", "when", "where", "why", "how", "which"],
    "Do/Does/Did Questions": ["do", "does", "did"],
    "Doubt/Clarification": [
        "is it possible", "do you mean", "are you saying", 
        "does this imply", "could you clarify", "what if", 
        "does that mean", "is there a reason"
    ],
    "Contextual Questions": [
        "is there", "will this", "does this", "should we", 
        "could this", "is this", "can this"
    ],
    "Exploration/Inquiry": ["explain", "define", "elaborate", "clarify", "give details about", "go into depth about"],
    "Comparison Questions": ["how does this compare", "is this better", "which one is better", "what's the difference"],
    "Hypothetical Scenarios": ["what if", "imagine if", "suppose", "assuming that", "let's say"],
    "Problem-Solving Questions": ["how can we", "what's the solution", "how do we fix", "how do we solve"],
    "Confirmation Questions": ["is it true", "is this correct", "am I right", "does this match"],
    "Reasoning/Justification": ["why is", "what's the reason", "how come", "why does this"],
    "Permission/Request": ["can I", "could you", "may I", "is it okay if"],
    "Instruction/Procedure": ["how do I", "what's the process", "what steps", "how to"]
}

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/harshdarji/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
# Clean and normalize text
def clean_text(text):
    # Step 1: Keep ? in natural language questions by protecting it
    text = re.sub(r'(?<=[\w\s])\?(?=\s|$)', '<QUESTION>', text)

    # Step 2: Remove Markdown links but keep visible text
    text = re.sub(r'\[([^\]]+)\]\([^)]+\)', r'\1', text)

    # Step 3: Remove Markdown special characters
    markdown_chars = r'[*_~`#>\[\]\(\)!\\\-]'
    text = re.sub(markdown_chars, '', text)

    # Step 4: Remove escape characters like \n, \r, \t, etc.
    text = re.sub(r'[\n\r\t\f\v]', ' ', text)

    # Step 5: Remove all remaining question marks (in code, URLs, etc.)
    text = text.replace('?', '')

    # Step 6: Restore protected question marks
    text = text.replace('<QUESTION', '?')

    # Step 7: Collapse extra whitespace
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

# Detect if a comment is a question
def is_question(comment):
    if "?" in comment:
        tokens = word_tokenize(comment.lower())
        for category, keywords in categories.items():
            if any(keyword in tokens for keyword in keywords):
                return True
    return False

# Categorize a question based on keywords
def categorize_question(comment):
    tokens = word_tokenize(comment.lower())
    for category, keywords in categories.items():
        for keyword in keywords:
            if keyword in tokens:
                return category, keyword
    return "Uncategorized", None

# Process each individual comment
def process_comment(comment, q_count):
    comment = clean_text(comment)
    if is_question(comment):
        category, keyword = categorize_question(comment)
        if category != "Uncategorized":
            q_count[category][keyword] += 1
        return {"type": "Q", "category": category, "body": comment}
    else:
        return {"type": "Answer", "body": comment}

# Process entire JSON dataset
def process_json(file_path):
    #print(1) # DEBUG.
    # Detect encoding
    with open(file_path, 'rb') as f:
        raw_data = f.read()
        encoding = chardet.detect(raw_data)['encoding']
        if not encoding:
            encoding = 'utf-8'  # Fallback

    # Load the JSON with the detected encoding
    with open(file_path, 'r', encoding=encoding, errors='replace') as file:
        try:
            data = json.load(file)
        except json.JSONDecodeError as e:
            print(f"❌ Skipping file due to JSON decode error: {e}")
            return {}, defaultdict(lambda: defaultdict(int))

    categorized_data = {}
    q_count = defaultdict(lambda: defaultdict(int))

    for repo_url, issues in data.items():
        categorized_data[repo_url] = []

        for issue in issues:
            try:
                issue_url = issue["issue_url"]
                issue_body = issue["issue_body"]
                comments = issue["comments"]
            except Exception:
                continue  # ❌ Skip the entire issue if it's malformed

            processed_comments = []
            for comment in comments:
                try:
                    processed_comment = process_comment(comment["body"], q_count)
                    processed_comment["comment_url"] = comment["comment_url"]
                    processed_comment["issue_url"] = issue_url
                    processed_comment["user"] = comment["user"]
                    processed_comment["created_at"] = comment["created_at"]
                    processed_comment["reactions"] = comment.get("reactions", [])
                    processed_comments.append(processed_comment)
                except Exception:
                    continue  # ❌ Skip the comment if it's malformed

            categorized_data[repo_url].append({
                "issue_url": issue_url,
                "issue_body": issue_body,
                "comments": processed_comments
            })

    return categorized_data, q_count

# Save the results to a new JSON file
def save_categorized_data(data, output_file):
    with open(output_file, 'w') as file:
        json.dump(data, file, indent=2)
    print(f"\n✅ Categorized data saved to: {output_file}")

# Print a summary of question counts by category
def display_q_counts(q_count):
    print("\n📊 Question Counts by Category and Sub-Category:")
    for category, keywords in q_count.items():
        print(f"\n{category}:")
        for keyword, count in keywords.items():
            print(f"  {keyword} = {count}")

In [6]:
# Main entry point
def main():
    input_file = "issuesWithComments.json"
    output_file = "/Users/harshdarji/Documents/Research-Internship-II/revised-ra2-iclr/categorizedCommentsV3.json"

    if not os.path.exists(input_file):
        print(f"❌ Input file {input_file} not found.")
        return

    try:
        #print("Trying...")
        categorized_data, q_count = process_json(input_file)
        save_categorized_data(categorized_data, output_file)
        display_q_counts(q_count)
    except Exception as e:
        print(f"❌ An error occurred: {e}")

In [7]:
main()


✅ Categorized data saved to: /Users/harshdarji/Documents/Research-Internship-II/revised-ra2-iclr/categorizedCommentsV3.json

📊 Question Counts by Category and Sub-Category:

Modal Verbs:
  can = 4158
  could = 1290
  would = 1126
  will = 393
  should = 604
  might = 135
  must = 30
  may = 161
  shall = 7

Interrogatives:
  how = 292
  what = 722
  who = 20
  which = 133
  why = 155
  where = 91
  when = 208

Do/Does/Did Questions:
  does = 197
  do = 328
  did = 185

Hypothetical Scenarios:
  suppose = 2

Exploration/Inquiry:
  define = 4
  clarify = 2
  explain = 1


In [1]:
import json

def load_orchid_dataset(file_path):
    """Load Orchid dataset from JSONL file"""
    tasks = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                tasks.append(json.loads(line))
    return tasks

# Load dataset
tasks = load_orchid_dataset('Orchid.jsonl')
print(f"Loaded {len(tasks)} tasks")

Loaded 164 tasks


In [ ]:
print(type(tasks))

In [3]:
print(*tasks[0].keys(), sep="\n")

name
entry_point
prompt
solution
test_case
Lexical_prompt
Lexical_ambiguity_explanation
Semantic_prompt
Semantic_ambiguity_explanation
Syntactic_prompt
Syntactic_ambiguity_explanation
Vagueness_prompt
Vagueness_ambiguity_explanation


In [4]:
print(*tasks[0].values(), sep="\n")

HumanEval/0
has_close_elements
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    """

from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """
    for idx, elem in enumerate(numbers):
        for idx2, elem2 in enumerate(numbers):
            if idx != idx2:
                distance = abs(elem - elem2)
                if distance < threshold:
                    return True

    return False

[{'input': '[1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3', 'output': 'True', 'relation': '=='}, {'input': '[1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.05', 'output': 'Fal

In [2]:
for k, v in tasks[1].items():
    print(f"====================KEY====================")
    print(k)
    print(f"====================VALUE=====================")
    print(v)
    print()

====================KEY====================
name
====================VALUE=====================
HumanEval/1

====================KEY====================
entry_point
====================VALUE=====================
separate_paren_groups

====================KEY====================
prompt
====================VALUE=====================
from typing import List


def separate_paren_groups(paren_string: str) -> List[str]:
    """ Input to this function is a string containing multiple groups of nested parentheses. Your goal is to
    separate those group into separate strings and return the list of those.
    Separate groups are balanced (each open brace is properly closed) and not nested within each other
    Ignore any spaces in the input string.
    """


====================KEY====================
solution
====================VALUE=====================
from typing import List


def separate_paren_groups(paren_string: str) -> List[str]:
    """ Input to this function is a string containing m

In [1]:
!huggingface-cli login --token "***REMOVED-HF-TOKEN***"

⚠️  Warning: 'huggingface-cli login' is deprecated. Use 'hf auth login' instead.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `question-asking-finetune` has been saved to /Users/harshdarji/.cache/huggingface/stored_tokens
Your token has been saved to /Users/harshdarji/.cache/huggingface/token
Login successful.
The current active token is: `question-asking-finetune`


In [6]:
import os, json, csv
import numpy as np
import torch

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from textwrap import shorten

# =========================
# CONFIG
# =========================
jsonl_file = "orchid.jsonl"
threshold = 0.5  # ambiguous if P(class3)+P(class4) >= threshold

# First existing path wins
ckpt_candidates = [
    "/Users/harshdarji/Documents/Research-Internship-II/revised-ra2-iclr/content/distilbert_ambiguity_ckpts/checkpoint-7488",
    "/content/distilbert_ambiguity_ckpts/checkpoint-7488",
    "distilbert_ambiguity_ckpts/checkpoint-7488",
]

# =========================
# LOAD JSONL -> PROMPTS ONLY
# =========================
def safe_strip(x):
    return x.strip() if isinstance(x, str) else x

prompts = []
with open(jsonl_file, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        row = json.loads(line)
        text = safe_strip(row.get("prompt", ""))
        if text:
            prompts.append(text)

if not prompts:
    raise ValueError("No 'prompt' texts found in orchid.jsonl.")

print(f"Loaded {len(prompts)} test examples from 'prompt' only.")

# =========================
# RESOLVE CHECKPOINT
# =========================
ckpt_path = next((p for p in ckpt_candidates if os.path.isdir(p)), None)
if ckpt_path is None:
    raise SystemExit("Aborting: no valid local checkpoint directory found in ckpt_candidates.")
print(f"Loading fine-tuned checkpoint from: {ckpt_path}")

# =========================
# TOKENIZER + MODEL
# =========================
try:
    tokenizer = DistilBertTokenizerFast.from_pretrained(ckpt_path)
except Exception:
    tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

model = DistilBertForSequenceClassification.from_pretrained(ckpt_path)
model.eval()

# =========================
# INFERENCE (CPU to avoid MPS issues)
# =========================
device = torch.device("cpu")
model.to(device)

enc = tokenizer(prompts, padding=True, truncation=True, return_tensors="pt")
enc = {k: v.to(device) for k, v in enc.items()}

with torch.no_grad():
    out = model(**enc)
logits = out.logits  # (N, C)

# =========================
# 4-class -> binary collapse
# classes 1–2 (idx 0,1): unambiguous (0)
# classes 3–4 (idx 2,3): ambiguous   (1)
# =========================
num_labels = logits.shape[1]
if num_labels == 4:
    probs = torch.softmax(logits, dim=-1).cpu().numpy()       # (N,4)
    p_amb = probs[:, 2] + probs[:, 3]
elif num_labels == 2:
    probs = torch.softmax(logits, dim=-1).cpu().numpy()       # (N,2)
    p_amb = probs[:, 1]
elif num_labels == 1:
    p_amb = torch.sigmoid(logits.squeeze(-1)).cpu().numpy()   # (N,)
    # Create a 2-col "probs" just for top-class display
    probs = np.stack([1 - p_amb, p_amb], axis=1)
else:
    raise ValueError(f"Unexpected num_labels: {num_labels}")

pred_bin = (p_amb >= threshold).astype(int)

# Top 4-class prediction (for display)
if num_labels >= 2:
    probs_for_top = probs
    pred_idx = probs_for_top.argmax(axis=1)          # 0..C-1
    pred_class_1based = (pred_idx + 1).tolist()      # 1..C
    top_prob = probs_for_top[np.arange(len(probs_for_top)), pred_idx]
else:
    pred_class_1based = [1] * len(prompts)
    top_prob = np.ones(len(prompts))

# =========================
# METRICS (ambiguous=1)
# =========================
y_true = np.ones(len(prompts), dtype=int)  # your test set is all ambiguous
acc = accuracy_score(y_true, pred_bin)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, pred_bin, average="binary", zero_division=0)
cm = confusion_matrix(y_true, pred_bin, labels=[0, 1])

print(f"\nBinary evaluation (ambiguous=1) with threshold={threshold}:")
print(f"Accuracy : {acc:.4f}  Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}")
print("Confusion matrix [rows=true, cols=pred]:\n", cm)

# Inspect a few false negatives
fn_indices = np.where((y_true == 1) & (pred_bin == 0))[0].tolist()
print(f"\nFalse negatives: {len(fn_indices)}")
for i in fn_indices[:10]:
    print(f"  idx={i}, P(ambiguous)={p_amb[i]:.3f}")

# =========================
# DISPLAY TABLE (per-example)
# =========================
print(f"\n{'i':>2} | {'pred4c':>6} | {'top_prob':>8} | {'p_amb':>8} | {'bin':>3} | prompt")
print("-"*100)
for i, (pc, tp, pa, pb, txt) in enumerate(zip(pred_class_1based, top_prob, p_amb, pred_bin, prompts)):
    print(f"{i:>2} | {pc:>6} | {tp:8.3f} | {pa:8.3f} | {pb:>3d} | {shorten(txt, width=80, placeholder='…')}")

amb_idxs = [i for i, b in enumerate(pred_bin) if b == 1]
unamb_idxs = [i for i, b in enumerate(pred_bin) if b == 0]
print("\nAmbiguous (bin=1) indices:", amb_idxs)
print("Unambiguous (bin=0) indices:", unamb_idxs)

# =========================
# SAVE PER-EXAMPLE PREDICTIONS
# =========================
with open("orchid_binary_preds.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow(["idx", "gold(ambiguous=1)", "p_ambiguous", "pred_label(0/1)"])
    for i, (pa, pb) in enumerate(zip(p_amb, pred_bin)):
        w.writerow([i, 1, float(pa), int(pb)])

print("\nSaved per-example predictions to orchid_binary_preds.csv")

Loaded 164 test examples from 'prompt' only.
Loading fine-tuned checkpoint from: /Users/harshdarji/Documents/Research-Internship-II/revised-ra2-iclr/content/distilbert_ambiguity_ckpts/checkpoint-7488

Binary evaluation (ambiguous=1) with threshold=0.5:
Accuracy : 0.0671  Precision: 1.0000  Recall: 0.0671  F1: 0.1257
Confusion matrix [rows=true, cols=pred]:
 [[  0   0]
 [153  11]]

False negatives: 153
  idx=0, P(ambiguous)=0.001
  idx=1, P(ambiguous)=0.000
  idx=2, P(ambiguous)=0.000
  idx=3, P(ambiguous)=0.000
  idx=4, P(ambiguous)=0.001
  idx=5, P(ambiguous)=0.141
  idx=6, P(ambiguous)=0.000
  idx=7, P(ambiguous)=0.007
  idx=8, P(ambiguous)=0.001
  idx=9, P(ambiguous)=0.000

 i | pred4c | top_prob |    p_amb | bin | prompt
----------------------------------------------------------------------------------------------------
 0 |      2 |    0.999 |    0.001 |   0 | from typing import List def has_close_elements(numbers: List[float], threshold:…
 1 |      2 |    0.999 |    0.000 |   0 |

In [5]:
# -------------------------
# SAMPLE PROMPTS (edit freely)
# -------------------------
device = torch.device("cpu")
model.to(device)

prompts = [
    # Clear / well-specified (unambiguous)
    "Write a Python function `is_palindrome(s: str) -> bool` that returns True if s reads the same forwards and backwards, ignoring case and non-alphanumeric characters. Include unit tests.",
    "Implement a function `two_sum(nums: List[int], target: int) -> Tuple[int,int]` returning 0-based indices of two distinct elements whose sum equals target. If none exist, raise ValueError.",
    "Given a sorted array of unique integers and a target, return the index of target using binary search or -1 if not found. O(log n) time.",

    # Ambiguous / underspecified
    "Make the code faster.",
    "Handle the edge cases in this parser.",
    "Build a class for caching results like we discussed.",
    "Create a solution for the parentheses problem.",
    "Fix the bug in my DFS implementation and optimize it.",
]

# -------------------------
# INFERENCE
# -------------------------
enc = tokenizer(prompts, padding=True, truncation=True, return_tensors="pt")
enc = {k: v.to(device) for k, v in enc.items()}

with torch.no_grad():
    out = model(**enc)
logits = out.logits  # (N, 4) expected
probs = torch.softmax(logits, dim=-1).cpu().numpy()  # (N, 4)

# Map 4-class -> binary:
#   classes 1,2 (indices 0,1) => unambiguous (0)
#   classes 3,4 (indices 2,3) => ambiguous (1)
p_amb = probs[:, 2] + probs[:, 3]
pred_bin = (p_amb >= threshold).astype(int)

pred_idx = probs.argmax(axis=1)                # 0..3
pred_class_1based = (pred_idx + 1).tolist()    # 1..4
top_prob = probs[np.arange(len(probs)), pred_idx]

# -------------------------
# DISPLAY
# -------------------------
from textwrap import shorten
print(f"{'i':>2} | {'pred4c':>6} | {'top_prob':>8} | {'p_amb':>8} | {'bin':>3} | prompt")
print("-"*100)
for i, (pc, tp, pa, pb, txt) in enumerate(zip(pred_class_1based, top_prob, p_amb, pred_bin, prompts)):
    print(f"{i:>2} | {pc:>6} | {tp:8.3f} | {pa:8.3f} | {pb:>3d} | {shorten(txt, width=80, placeholder='…')}")

# Optional: quickly see which ones were flagged as ambiguous
amb_idxs = [i for i, b in enumerate(pred_bin) if b == 1]
unamb_idxs = [i for i, b in enumerate(pred_bin) if b == 0]
print("\nAmbiguous (bin=1) indices:", amb_idxs)
print("Unambiguous (bin=0) indices:", unamb_idxs)

 i | pred4c | top_prob |    p_amb | bin | prompt
----------------------------------------------------------------------------------------------------
 0 |      2 |    0.989 |    0.011 |   0 | Write a Python function `is_palindrome(s: str) -> bool` that returns True if s…
 1 |      1 |    0.999 |    0.000 |   0 | Implement a function `two_sum(nums: List[int], target: int) -> Tuple[int,int]`…
 2 |      1 |    0.999 |    0.000 |   0 | Given a sorted array of unique integers and a target, return the index of…
 3 |      3 |    0.996 |    1.000 |   1 | Make the code faster.
 4 |      3 |    0.997 |    1.000 |   1 | Handle the edge cases in this parser.
 5 |      3 |    0.998 |    1.000 |   1 | Build a class for caching results like we discussed.
 6 |      4 |    0.996 |    1.000 |   1 | Create a solution for the parentheses problem.
 7 |      3 |    0.996 |    0.999 |   1 | Fix the bug in my DFS implementation and optimize it.

Ambiguous (bin=1) indices: [3, 4, 5, 6, 7]
Unambiguous (bin=0) i

In [9]:
import os, json, csv
import numpy as np
import torch

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from textwrap import shorten

# =========================
# CONFIG
# =========================
jsonl_file = "orchid.jsonl"
ckpt_candidates = [
    "/Users/harshdarji/Documents/Research-Internship-II/revised-ra2-iclr/content/distilbert_ambiguity_ckpts/checkpoint-7488",
    "/content/distilbert_ambiguity_ckpts/checkpoint-7488",
    "distilbert_ambiguity_ckpts/checkpoint-7488",
]

# =========================
# LOAD JSONL -> PROMPTS ONLY
# =========================
def safe_strip(x):
    return x.strip() if isinstance(x, str) else x

prompts = []
with open(jsonl_file, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        row = json.loads(line)
        text = safe_strip(row.get("prompt", ""))
        if text:
            prompts.append(text)

if not prompts:
    raise ValueError("No 'prompt' texts found in orchid.jsonl.")
print(f"Loaded {len(prompts)} test examples from 'prompt' only.")

# =========================
# RESOLVE CHECKPOINT
# =========================
ckpt_path = next((p for p in ckpt_candidates if os.path.isdir(p)), None)
if ckpt_path is None:
    raise SystemExit("Aborting: no valid local checkpoint directory found.")
print(f"Loading fine-tuned checkpoint from: {ckpt_path}")

# =========================
# TOKENIZER + MODEL
# =========================
try:
    tokenizer = DistilBertTokenizerFast.from_pretrained(ckpt_path)
except Exception:
    tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

model = DistilBertForSequenceClassification.from_pretrained(ckpt_path)
model.eval()

# =========================
# INFERENCE (CPU to avoid MPS quirks)
# =========================
device = torch.device("cpu")
model.to(device)

enc = tokenizer(prompts, padding=True, truncation=True, return_tensors="pt")
enc = {k: v.to(device) for k, v in enc.items()}

with torch.no_grad():
    out = model(**enc)

logits_t = out.logits  # torch.Tensor [N, C]
num_labels = logits_t.shape[1]

# ---- enforce 4-logit head ----
assert model.config.num_labels == 4, f"Checkpoint has num_labels={model.config.num_labels}, expected 4."
assert num_labels == 4, f"Model returned {num_labels} logits, expected 4."

# softmax probs and argmax class
probs = torch.softmax(logits_t, dim=-1).cpu().numpy()        # [N,4]
logits = logits_t.cpu().numpy()                              # [N,4] raw logits
pred_idx = probs.argmax(axis=1)                              # 0..3
pred_class_1based = (pred_idx + 1).tolist()                  # 1..4
top_prob = probs[np.arange(len(probs)), pred_idx]

# ----- collapse to binary via argmax mapping -----
# classes 1–2 (idx 0,1) -> unambiguous (0)
# classes 3–4 (idx 2,3) -> ambiguous   (1)
pred_bin = (pred_idx >= 2).astype(int)

# also keep a confidence-style number for ambiguity
p_amb = probs[:, 2] + probs[:, 3]                            # P(class3)+P(class4)

# =========================
# METRICS (if your whole set is ambiguous)
# =========================
y_true = np.ones(len(prompts), dtype=int)  # change if you have mixed labels later
acc = accuracy_score(y_true, pred_bin)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, pred_bin, average="binary", zero_division=0)
cm = confusion_matrix(y_true, pred_bin, labels=[0, 1])

print(f"\nBinary evaluation (ambiguous=1 from 4-class argmax):")
print(f"Accuracy : {acc:.4f}  Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}")
print("Confusion matrix [rows=true, cols=pred]:\n", cm)

# =========================
# PRINT RAW LOGITS PER EXAMPLE
# =========================
import numpy as np
np.set_printoptions(precision=4, suppress=True)

# If your 'logits' is a torch tensor in your script, make sure it's numpy:
logits_np = logits if isinstance(logits, np.ndarray) else logits.detach().cpu().numpy()

print("\nRaw logits per example (model outputs before softmax):")
for i, logit_row in enumerate(logits_np):  # shape: (N, 4)
    formatted = ", ".join(f"{x:.4f}" for x in logit_row)
    print(f"Example #{i}: [{formatted}]  p_amb={p_amb[i]:.4f}  top4c={pred_class_1based[i]}  bin={pred_bin[i]}")


# =========================
# DISPLAY TABLE (per-example)
# =========================
from textwrap import shorten

print(f"\n{'i':>3} | {'top4c':>6} | {'topProb':>8} | {'p1':>6} | {'p2':>6} | {'p3':>6} | {'p4':>6} | {'bin':>3} | prompt")
print("-"*130)
for i, (pc, tp, pb, p1, p2, p3, p4, txt) in enumerate(
    zip(pred_class_1based, top_prob, pred_bin, probs[:,0], probs[:,1], probs[:,2], probs[:,3], prompts)
):
    preview = shorten(txt, width=80, placeholder="…")  # <-- keyword arg
    print(f"{i:>3} | {pc:>6} | {tp:8.3f} | {p1:6.3f} | {p2:6.3f} | {p3:6.3f} | {p4:6.3f} | {pb:>3d} | {preview}")


# =========================
# SAVE PER-EXAMPLE (raw logits + probs)
# =========================
with open("orchid_4class_and_binary_preds.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow([
        "idx", "prompt",
        "logit_c1","logit_c2","logit_c3","logit_c4",
        "prob_c1","prob_c2","prob_c3","prob_c4",
        "top_4class","top_prob",
        "p_ambiguous(c3+c4)","bin_from_argmax(0/1)"
    ])
    for i, txt in enumerate(prompts):
        w.writerow([
            i, txt,
            float(logits[i,0]), float(logits[i,1]), float(logits[i,2]), float(logits[i,3]),
            float(probs[i,0]), float(probs[i,1]), float(probs[i,2]), float(probs[i,3]),
            int(pred_class_1based[i]), float(top_prob[i]),
            float(p_amb[i]), int(pred_bin[i])
        ])

print("\nSaved per-example predictions to orchid_4class_and_binary_preds.csv")


Loaded 164 test examples from 'prompt' only.
Loading fine-tuned checkpoint from: /Users/harshdarji/Documents/Research-Internship-II/revised-ra2-iclr/content/distilbert_ambiguity_ckpts/checkpoint-7488

Binary evaluation (ambiguous=1 from 4-class argmax):
Accuracy : 0.0732  Precision: 1.0000  Recall: 0.0732  F1: 0.1364
Confusion matrix [rows=true, cols=pred]:
 [[  0   0]
 [152  12]]

Raw logits per example (model outputs before softmax):
Example #0: [-1.7297, 5.5641, -1.9821, -7.2682]  p_amb=0.0005  top4c=2  bin=0
Example #1: [-1.5404, 5.6438, -2.0584, -7.2406]  p_amb=0.0005  top4c=2  bin=0
Example #2: [5.8945, -0.7864, -3.6127, -6.1318]  p_amb=0.0001  top4c=1  bin=0
Example #3: [5.2443, -0.2930, -3.1666, -5.9166]  p_amb=0.0002  top4c=1  bin=0
Example #4: [0.0210, 4.5895, -1.9921, -7.6947]  p_amb=0.0014  top4c=2  bin=0
Example #5: [-3.8232, 2.6349, 0.8295, -6.1142]  p_amb=0.1411  top4c=2  bin=0
Example #6: [5.0764, 1.1006, -3.7686, -7.2337]  p_amb=0.0001  top4c=1  bin=0
Example #7: [-1.8

In [10]:
import os, json, csv
import numpy as np
import torch

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from textwrap import shorten

# =========================
# CONFIG
# =========================
jsonl_file = "orchid.jsonl"
# First existing path wins
ckpt_candidates = [
    "/Users/harshdarji/Documents/Research-Internship-II/revised-ra2-iclr/content/distilbert_ambiguity_ckpts/checkpoint-7488",
    "/content/distilbert_ambiguity_ckpts/checkpoint-7488",
    "distilbert_ambiguity_ckpts/checkpoint-7488",
]
PREVIEW_N = 15  # how many rows to preview in the "raw logits + probs" section

# =========================
# LOAD JSONL -> PROMPTS ONLY
# =========================
def safe_strip(x):
    return x.strip() if isinstance(x, str) else x

prompts = []
with open(jsonl_file, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        row = json.loads(line)
        text = safe_strip(row.get("prompt", ""))
        if text:
            prompts.append(text)

if not prompts:
    raise ValueError("No 'prompt' texts found in orchid.jsonl.")
print(f"Loaded {len(prompts)} test examples from 'prompt' only.")

# =========================
# RESOLVE CHECKPOINT
# =========================
ckpt_path = next((p for p in ckpt_candidates if os.path.isdir(p)), None)
if ckpt_path is None:
    raise SystemExit("Aborting: no valid local checkpoint directory found in ckpt_candidates.")
print(f"Loading fine-tuned checkpoint from: {ckpt_path}")

# =========================
# TOKENIZER + MODEL
# =========================
try:
    tokenizer = DistilBertTokenizerFast.from_pretrained(ckpt_path)
except Exception:
    tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

model = DistilBertForSequenceClassification.from_pretrained(ckpt_path)
model.eval()

# Enforce a 4-class head
assert model.config.num_labels == 4, f"Checkpoint has num_labels={model.config.num_labels}, expected 4."

# =========================
# INFERENCE (CPU to avoid MPS quirks)
# =========================
device = torch.device("cpu")
model.to(device)

enc = tokenizer(prompts, padding=True, truncation=True, return_tensors="pt")
enc = {k: v.to(device) for k, v in enc.items()}

with torch.no_grad():
    out = model(**enc)

logits_t = out.logits  # torch.Tensor [N, 4] (raw scores)
assert logits_t.shape[1] == 4, f"Model returned {logits_t.shape[1]} logits, expected 4."
logits = logits_t.cpu().numpy()

# Softmax -> probabilities in [0,1], rows sum to 1
probs = torch.softmax(logits_t, dim=-1).cpu().numpy()   # shape (N, 4)

# Top 4-class prediction and probability
pred_idx = probs.argmax(axis=1)                # 0..3
pred_class_1based = (pred_idx + 1).tolist()    # 1..4
top_prob = probs[np.arange(len(probs)), pred_idx]

# Collapse to binary by ARGMAX mapping: {1,2}→0, {3,4}→1
pred_bin = (pred_idx >= 2).astype(int)

# For display: P(ambiguous) = P(class3) + P(class4)
p_amb = probs[:, 2] + probs[:, 3]

# =========================
# METRICS (assume all prompts are ambiguous)
# =========================
y_true = np.ones(len(prompts), dtype=int)  # change if you later have mixed labels
acc = accuracy_score(y_true, pred_bin)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, pred_bin, average="binary", zero_division=0)
cm = confusion_matrix(y_true, pred_bin, labels=[0, 1])

print(f"\nBinary evaluation (ambiguous=1 from 4-class argmax):")
print(f"Accuracy : {acc:.4f}  Precision: {prec:.4f}  Recall: {rec:.4f}  F1: {f1:.4f}")
print("Confusion matrix [rows=true, cols=pred]:\n", cm)

# Inspect a few false negatives (model said unambiguous on an ambiguous item)
fn_indices = np.where((y_true == 1) & (pred_bin == 0))[0].tolist()
print(f"\nFalse negatives: {len(fn_indices)}")
for i in fn_indices[:min(len(fn_indices), 10)]:
    print(f"  idx={i}, p_amb={p_amb[i]:.3f}")

# =========================
# PREVIEW: Raw logits + softmax probabilities
# =========================
print("\nRaw logits + softmax probabilities (first {} examples):".format(min(PREVIEW_N, len(prompts))))
for i in range(min(PREVIEW_N, len(prompts))):
    l = [float(x) for x in logits[i]]
    p = [float(x) for x in probs[i]]
    print(
        f"Example #{i}: logits={np.round(l, 4)}  "
        f"probs={np.round(p, 4)} (sum={probs[i].sum():.3f})  "
        f"p_amb={p_amb[i]:.4f}  top4c={pred_class_1based[i]}  bin={pred_bin[i]}"
    )

# =========================
# DISPLAY TABLE (per-example)
# =========================
print(f"\n{'i':>3} | {'top4c':>6} | {'topProb':>8} | {'p1':>6} | {'p2':>6} | {'p3':>6} | {'p4':>6} | {'bin':>3} | prompt")
print("-"*130)
for i, (pc, tp, pb, p1, p2, p3, p4, txt) in enumerate(
    zip(pred_class_1based, top_prob, pred_bin, probs[:,0], probs[:,1], probs[:,2], probs[:,3], prompts)
):
    preview = shorten(txt, width=80, placeholder="…")
    print(f"{i:>3} | {pc:>6} | {tp:8.3f} | {p1:6.3f} | {p2:6.3f} | {p3:6.3f} | {p4:6.3f} | {pb:>3d} | {preview}")

# =========================
# SAVE PER-EXAMPLE (raw logits + probs + binary)
# =========================
with open("orchid_4class_and_binary_preds.csv", "w", encoding="utf-8", newline="") as f:
    w = csv.writer(f)
    w.writerow([
        "idx", "prompt",
        "logit_c1","logit_c2","logit_c3","logit_c4",
        "prob_c1","prob_c2","prob_c3","prob_c4",
        "top_4class","top_prob",
        "p_ambiguous(c3+c4)","bin_from_argmax(0/1)"
    ])
    for i, txt in enumerate(prompts):
        w.writerow([
            i, txt,
            float(logits[i,0]), float(logits[i,1]), float(logits[i,2]), float(logits[i,3]),
            float(probs[i,0]),  float(probs[i,1]),  float(probs[i,2]),  float(probs[i,3]),
            int(pred_class_1based[i]), float(top_prob[i]),
            float(p_amb[i]), int(pred_bin[i])
        ])

print("\nSaved per-example predictions to orchid_4class_and_binary_preds.csv")


Loaded 164 test examples from 'prompt' only.
Loading fine-tuned checkpoint from: /Users/harshdarji/Documents/Research-Internship-II/revised-ra2-iclr/content/distilbert_ambiguity_ckpts/checkpoint-7488

Binary evaluation (ambiguous=1 from 4-class argmax):
Accuracy : 0.0732  Precision: 1.0000  Recall: 0.0732  F1: 0.1364
Confusion matrix [rows=true, cols=pred]:
 [[  0   0]
 [152  12]]

False negatives: 152
  idx=0, p_amb=0.001
  idx=1, p_amb=0.000
  idx=2, p_amb=0.000
  idx=3, p_amb=0.000
  idx=4, p_amb=0.001
  idx=5, p_amb=0.141
  idx=6, p_amb=0.000
  idx=7, p_amb=0.007
  idx=8, p_amb=0.001
  idx=9, p_amb=0.000

Raw logits + softmax probabilities (first 15 examples):
Example #0: logits=[-1.7297  5.5641 -1.9821 -7.2682]  probs=[0.0007 0.9988 0.0005 0.    ] (sum=1.000)  p_amb=0.0005  top4c=2  bin=0
Example #1: logits=[-1.5404  5.6438 -2.0584 -7.2406]  probs=[0.0008 0.9988 0.0005 0.    ] (sum=1.000)  p_amb=0.0005  top4c=2  bin=0
Example #2: logits=[ 5.8945 -0.7864 -3.6127 -6.1318]  probs=[0.